# Ejercicio a)

Se pide crear un script en python que dada la trayectoría ground-truth (timestamp, x,
y, z, qw, qx, qy, qz) (primeras 8 columnas del archivo mav0/state_groundtruth_estimate0/data
.csv) genere el camino ground-truth de la cámara izquierda, esté debe estar dado en el sistema de
coordenadas de la cámara izquierda inicial. Para esto deberá utilizar las transformaciones provistas
en el dataset.

* Lo pedido es un proceso similar al del ejercicio 4.
* Allí, podíamos obtener fácilmente todos los ${}^W \xi_{C_i}$ (la primera pregunta)}
* y en base a eso, obtener los ${}^W \xi_{B_i}$
* En este ejercicio, el objetivo es ${}^{C_0} \xi_{C_i}$

En este ejercicio, ya tenemos los ${}^W \xi_{B_i}$, nos lo dice el propio enunciado.

Por otro lado, tenemos la transformación ${}^B \xi_{C}$ en el archivo `./cam0/data/sensor.yaml`. (Vamos a tomar a $C$ como la cámara izquierda, la camara derecha no debería mecionarse en principio)

```py
# ./cam0/data/sensor.yaml
T_BS:
  cols: 4
  rows: 4
  data: [0.0148655429818, -0.999880929698, 0.00414029679422, -0.0216401454975,
         0.999557249008, 0.0149672133247, 0.025715529948, -0.064676986768,
        -0.0257744366974, 0.00375618835797, 0.999660727178, 0.00981073058949,
         0.0, 0.0, 0.0, 1.0]
```

Así, tenemos:

$$
{}^B \xi_{C}
=
\begin{bmatrix}
0.0148655429818 & -0.999880929698 & 0.00414029679422 & -0.0216401454975 \\
0.999557249008 & 0.0149672133247 & 0.025715529948 & -0.064676986768 \\
-0.0257744366974 & 0.00375618835797 & 0.999660727178 & 0.00981073058949 \\
0.0 & 0.0 & 0.0 & 1.0
\end{bmatrix}
$$

Razonando de manera similar al ejercicio 4, tenemos,

$$
{}^W \xi_{C_i}
=
{}^W \xi_{B_i} \cdot {}^{B_i} \xi_{C_i}
=
{}^W \xi_{B_i} \cdot {}^B \xi_{C}
$$

Se nos pide expresamente en el enunciado:

> Genere el camino ground-truth de la cámara izquierda, **esté debe estar dado en el sistema de coordenadas de la cámara izquierda inicial.**

Por lo tanto, deberíamos obtener

$$
{}^{C_0} \xi_{C_i} \forall i \in N
$$

Notemos que

$$
{}^{C_0} \xi_{C_i}
=
{}^{C_0} \xi_{W} \cdot {}^{W} \xi_{C_i}
=
({}^{W} \xi_{C_0})^{-1} \cdot {}^{W} \xi_{C_i}
$$

## Desarrollo

In [21]:
import csv
import yaml
import numpy as np
from transforms3d.quaternions import quat2mat, mat2quat

Vamos a llamar `T_BS` a ${}^{B} \xi_{C}$, por convención del dataset y por facilidad

In [22]:
with open('mav0/cam0/sensor.yaml', 'r') as f:
    cam0YAML = yaml.safe_load(f)

T_BS_data = cam0YAML['T_BS']['data']
T_BS = np.array(T_BS_data).reshape(4, 4)

print(T_BS)

[[ 0.0149 -0.9999  0.0041 -0.0216]
 [ 0.9996  0.015   0.0257 -0.0647]
 [-0.0258  0.0038  0.9997  0.0098]
 [ 0.      0.      0.      1.    ]]


In [23]:
gt_path = 'mav0/state_groundtruth_estimate0/data.csv'
ground_truth = []

with open(gt_path, 'r') as f:
    reader = csv.reader(f)
    header = next(reader)
    for row in reader:
        tmstmp = int(row[0])
        x, y, z =   float(row[1]),\
                    float(row[2]), \
                    float(row[3])
        qw, qx, qy, qz = float(row[4]), float(row[5]), float(row[6]), float(row[7])

        R_i = quat2mat([qw, qx, qy, qz])
        t_i = np.array([x, y, z])

        W_to_Bi = np.eye(4)
        W_to_Bi[:3, :3] = R_i
        W_to_Bi[:3, 3] = t_i

        ground_truth.append((tmstmp, W_to_Bi))

print(f"Se leyeron {len(ground_truth)} poses")

        


Se leyeron 36382 poses


In [24]:
camera_path_world = []

for tmstmp, W_to_Bi in ground_truth:
    W_to_Ci = W_to_Bi @ T_BS
    camera_path_world.append((tmstmp, W_to_Ci))

In [25]:
_, W_to_C0 = camera_path_world[0]
W_to_C0_inv = np.linalg.inv(W_to_C0)

In [26]:
camera_path_c0 = []

for tmstmp, W_to_Ci in camera_path_world:
    C0_to_Ci = W_to_C0_inv @ W_to_Ci
    camera_path_c0.append((tmstmp, C0_to_Ci))

In [27]:
# print(camera_path_c0)

In [28]:
np.set_printoptions(precision=4, suppress=True)

for tmstmp, C0_to_Ci in camera_path_c0[:5]:
    print(f"timestamp: {tmstmp}")
    print(C0_to_Ci)
    print("-" * 40)

timestamp: 1403636580838555648
[[ 1. -0.  0.  0.]
 [ 0.  1. -0.  0.]
 [ 0. -0.  1. -0.]
 [ 0.  0.  0.  1.]]
----------------------------------------
timestamp: 1403636580843555328
[[ 1.      0.0006  0.0011  0.0001]
 [-0.0006  1.     -0.0015 -0.0037]
 [-0.0011  0.0015  1.     -0.0014]
 [ 0.      0.      0.      1.    ]]
----------------------------------------
timestamp: 1403636580848555520
[[ 1.      0.0012  0.0022  0.0002]
 [-0.0012  1.     -0.0031 -0.0074]
 [-0.0022  0.0031  1.     -0.0028]
 [ 0.      0.      0.      1.    ]]
----------------------------------------
timestamp: 1403636580853555456
[[ 1.      0.0019  0.0033  0.0004]
 [-0.0019  1.     -0.0047 -0.0112]
 [-0.0033  0.0047  1.     -0.0041]
 [ 0.      0.      0.      1.    ]]
----------------------------------------
timestamp: 1403636580858555648
[[ 1.      0.0026  0.0045  0.0005]
 [-0.0026  1.     -0.0063 -0.0149]
 [-0.0046  0.0063  1.     -0.0055]
 [ 0.      0.      0.      1.    ]]
----------------------------------------